# Events Bronze Layer Ingestion

This notebook ingests StatsBomb event data from raw JSON files and transforms it into a bronze layer with the following schema:

## Bronze Schema
* **match_id** (BIGINT) - Extracted from filename
* **event_id** (STRING) - UUID for each event
* **index** (INT) - Event sequence number
* **period** (INT) - Match period (1st half, 2nd half, etc.)
* **minute** (INT) - Minute of the match
* **second** (INT) - Second within the minute
* **timestamp** (STRING) - Original timestamp from source
* **event_type_id** (INT) - Event type identifier
* **event_type_name** (STRING) - Event type name (Pass, Shot, etc.)
* **team_id** (INT) - Team identifier
* **team_name** (STRING) - Team name
* **player_id** (INT) - Player identifier (nullable)
* **player_name** (STRING) - Player name (nullable)
* **location_x** (DOUBLE) - X coordinate (nullable)
* **location_y** (DOUBLE) - Y coordinate (nullable)
* **raw_json** (STRING) - Full original event JSON for schema flexibility
* **ingestion_ts** (TIMESTAMP) - Ingestion timestamp

## Data Volume
* **~1.5M events** across **416 matches**
* **33 event types** including Pass, Ball Receipt, Carry, Pressure, etc.

In [0]:
import sys
sys.path.append('/Workspace/Users/pawanvirat32@gmail.com/MatchPulse')

from config.paths import EVENTS_RAW, EVENTS_BRONZE

print(f"Reading matches from: {EVENTS_RAW}")
print(f"Will write to: {EVENTS_BRONZE}")

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, LongType, IntegerType, DoubleType, TimestampType

# Read raw events JSON with match_id from file path
df_raw = spark.read \
    .option("multiLine", "true") \
    .option("mode", "PERMISSIVE") \
    .json(EVENTS_RAW)

# Add match_id from file path using _metadata.file_path
df_with_match = df_raw.withColumn(
    "match_id",
    F.regexp_extract(F.col("_metadata.file_path"), r"/(\d+)\.json", 1).cast("bigint")
)

# Flatten and transform to bronze schema
df_bronze = df_with_match.select(
    F.col("match_id").cast("bigint").alias("match_id"),
    F.col("id").alias("event_id"),
    F.col("index").cast("int").alias("index"),
    F.col("period").cast("int").alias("period"),
    F.col("minute").cast("int").alias("minute"),
    F.col("second").cast("int").alias("second"),
    F.col("timestamp").alias("timestamp"),
    F.col("type.id").cast("int").alias("event_type_id"),
    F.col("type.name").alias("event_type_name"),
    F.col("team.id").cast("int").alias("team_id"),
    F.col("team.name").alias("team_name"),
    F.col("player.id").cast("int").alias("player_id"),
    F.col("player.name").alias("player_name"),
    F.col("location")[0].alias("location_x"),
    F.col("location")[1].alias("location_y"),
    F.to_json(F.struct(F.col("*"))).alias("raw_json"),
    F.current_timestamp().alias("ingestion_ts")
)

print(f"Total events to ingest: {df_bronze.count():,}")
print("\nBronze schema:")
df_bronze.printSchema()
print("\nSample records:")
display(df_bronze.limit(10))

In [0]:
# Write the bronze data to S3
df_bronze.write \
    .mode("overwrite") \
    .format("parquet") \
    .option("compression", "snappy") \
    .save(EVENTS_BRONZE)

print(f"Successfully wrote events to bronze layer: {EVENTS_BRONZE}")
print(f"Total records written: {df_bronze.count():,}")

In [0]:
# Read back from bronze to verify
df_verify = spark.read.parquet(EVENTS_BRONZE)

print("Bronze layer statistics:")
print(f"Total events: {df_verify.count():,}")
print(f"Unique matches: {df_verify.select('match_id').distinct().count():,}")
print(f"Unique event types: {df_verify.select('event_type_name').distinct().count():,}")
print(f"Date range: {df_verify.agg(F.min('ingestion_ts'), F.max('ingestion_ts')).collect()[0]}")

print("\nEvent type distribution (top 10):")
df_verify.groupBy('event_type_name') \
    .count() \
    .orderBy(F.desc('count')) \
    .limit(10) \
    .show(truncate=False)

print("\nSample bronze records:")
display(df_verify.limit(5))